# .h5ad File Summary
This notebook loads an `.h5ad` file and prints a comprehensive summary of all columns, data types, and structure.

In [2]:
import anndata as ad
import pandas as pd

# ---- UPDATE THIS PATH ----
H5AD_PATH = "data/hnc_myeloid_2021.h5ad"
# --------------------------

adata = ad.read_h5ad(H5AD_PATH)
print(adata)

AnnData object with n_obs × n_vars = 26444 × 23630
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'seurat_clusters', 'RNA_snn_res.0.5', 'RNA_snn_res.0.6', 'RNA_snn_res.0.7', 'RNA_snn_res.0.8', 'RNA_snn_res.0.9', 'RNA_snn_res.1', 'RNA_snn_res.1.1', 'RNA_snn_res.1.2', 'RNA_snn_res.1.3', 'RNA_snn_res.1.4', 'RNA_snn_res.1.5', 'RNA_snn_res.1.6', 'RNA_snn_res.1.7', 'RNA_snn_res.1.8', 'RNA_snn_res.1.9', 'RNA_snn_res.2', 'tissue', 'is_HD', 'global.cluster', 'global.cluster2', 'hpv_status', 'RNA_snn_res.2.1', 'RNA_snn_res.2.2', 'RNA_snn_res.2.3', 'RNA_snn_res.2.4', 'RNA_snn_res.2.5', 'RNA_snn_res.2.6', 'RNA_snn_res.2.7', 'RNA_snn_res.2.8', 'RNA_snn_res.2.9', 'RNA_snn_res.3', 'tissue_hpv', 'global.cluster3', 'global.cluster4'
    obsm: 'X_harmony', 'X_pca', 'X_umap'


## 1. Shape & Basics

In [3]:
print(f"Number of observations (cells): {adata.n_obs}")
print(f"Number of variables (genes):    {adata.n_vars}")
print(f"\nX matrix type:  {type(adata.X).__name__}")
if hasattr(adata.X, 'dtype'):
    print(f"X matrix dtype: {adata.X.dtype}")
if hasattr(adata.X, 'shape'):
    print(f"X matrix shape: {adata.X.shape}")

Number of observations (cells): 26444
Number of variables (genes):    23630

X matrix type:  csr_matrix
X matrix dtype: int64
X matrix shape: (26444, 23630)


## 2. Observation metadata (`adata.obs`)

In [4]:
obs_summary = pd.DataFrame({
    'dtype': adata.obs.dtypes,
    'n_unique': [adata.obs[c].nunique() for c in adata.obs.columns],
    'n_missing': adata.obs.isnull().sum(),
    'example_values': [adata.obs[c].dropna().unique()[:5].tolist() for c in adata.obs.columns]
})
print(f"adata.obs — {adata.obs.shape[0]} rows x {adata.obs.shape[1]} columns\n")
display(obs_summary)

adata.obs — 26444 rows x 38 columns



,dtype,n_unique,n_missing,example_values
orig.ident,category,63,0,"[GSM4138110, GSM4138111, GSM4138112, GSM413811..."
nCount_RNA,int64,10812,0,"[1077, 2351, 946, 1088, 1217]"
nFeature_RNA,int64,3709,0,"[550, 992, 455, 556, 618]"
seurat_clusters,int64,27,0,"[3, 4, 2, 0, 9]"
RNA_snn_res.0.5,int64,15,0,"[2, 21, 6, 9, 0]"
RNA_snn_res.0.6,int64,15,0,"[2, 21, 6, 8, 0]"
RNA_snn_res.0.7,int64,16,0,"[2, 23, 17, 10, 0]"
RNA_snn_res.0.8,int64,16,0,"[2, 11, 27, 19, 13]"
RNA_snn_res.0.9,int64,14,0,"[2, 9, 26, 19, 13]"
RNA_snn_res.1,int64,17,0,"[1, 2, 3, 4, 5]"


In [7]:
adata.obs[["global.cluster4"]].to_csv("hnc_myeloid_globalcluster4.csv")

In [12]:
adata.obs[["global.cluster4"]].value_counts()

global.cluster4
Mono_CD14          7518
Mono_CD16          2503
Mac_IL1Bint        1842
Mono_CD14_IL1B     1801
Mono_CD14_THBS1    1783
Mono_Int           1684
cDC2_CD33          1319
Mac_CXCL9          1225
DC_pDC             1218
Mono_CD14_ID1      1154
Mono_TIL            988
cDC2_CD1C           966
Mac_IL1B            774
Mac_SPP1            573
mregDC_LAMP3        482
Mast                313
cDC1_CLEC9A         301
Name: count, dtype: int64

## 3. Variable metadata (`adata.var`)

In [4]:
var_summary = pd.DataFrame({
    'dtype': adata.var.dtypes,
    'n_unique': [adata.var[c].nunique() for c in adata.var.columns],
    'n_missing': adata.var.isnull().sum(),
    'example_values': [adata.var[c].dropna().unique()[:5].tolist() for c in adata.var.columns]
})
print(f"adata.var — {adata.var.shape[0]} rows x {adata.var.shape[1]} columns\n")
display(var_summary)

adata.var — 23630 rows x 0 columns



,dtype,n_unique,n_missing,example_values


## 4. Unstructured metadata (`adata.uns`)

In [5]:
if adata.uns:
    for key in sorted(adata.uns.keys()):
        val = adata.uns[key]
        print(f"  {key:30s}  type={type(val).__name__}")
else:
    print("  (empty)")

  (empty)


## 5. Obsm (embeddings / dim-reductions)

In [6]:
if adata.obsm:
    for key in sorted(adata.obsm.keys()):
        arr = adata.obsm[key]
        dtype = arr.dtype if hasattr(arr, 'dtype') else type(arr).__name__
        shape = arr.shape if hasattr(arr, 'shape') else 'N/A'
        print(f"  {key:30s}  shape={shape}  dtype={dtype}")
else:
    print("  (empty)")

  X_harmony                       shape=(26444, 100)  dtype=float64
  X_pca                           shape=(26444, 50)  dtype=float64
  X_umap                          shape=(26444, 2)  dtype=float64


## 6. Varm (variable embeddings)

In [7]:
if adata.varm:
    for key in sorted(adata.varm.keys()):
        arr = adata.varm[key]
        dtype = arr.dtype if hasattr(arr, 'dtype') else type(arr).__name__
        shape = arr.shape if hasattr(arr, 'shape') else 'N/A'
        print(f"  {key:30s}  shape={shape}  dtype={dtype}")
else:
    print("  (empty)")

  (empty)


## 7. Obsp (pairwise observation annotations)

In [8]:
if adata.obsp:
    for key in sorted(adata.obsp.keys()):
        arr = adata.obsp[key]
        dtype = arr.dtype if hasattr(arr, 'dtype') else type(arr).__name__
        shape = arr.shape if hasattr(arr, 'shape') else 'N/A'
        print(f"  {key:30s}  shape={shape}  dtype={dtype}")
else:
    print("  (empty)")

  (empty)


## 8. Varp (pairwise variable annotations)

In [9]:
if adata.varp:
    for key in sorted(adata.varp.keys()):
        arr = adata.varp[key]
        dtype = arr.dtype if hasattr(arr, 'dtype') else type(arr).__name__
        shape = arr.shape if hasattr(arr, 'shape') else 'N/A'
        print(f"  {key:30s}  shape={shape}  dtype={dtype}")
else:
    print("  (empty)")

  (empty)


## 9. Layers

In [10]:
if adata.layers:
    for key in sorted(adata.layers.keys()):
        arr = adata.layers[key]
        dtype = arr.dtype if hasattr(arr, 'dtype') else type(arr).__name__
        shape = arr.shape if hasattr(arr, 'shape') else 'N/A'
        print(f"  {key:30s}  shape={shape}  dtype={dtype}")
else:
    print("  (empty)")

  (empty)


## 10. Raw layer

In [11]:
if adata.raw is not None:
    print(f"  raw.X shape: {adata.raw.X.shape}")
    print(f"  raw.X dtype: {adata.raw.X.dtype}")
    print(f"  raw.var columns: {list(adata.raw.var.columns)}")
    print(f"  raw n_vars: {adata.raw.n_vars}")
else:
    print("  (no raw layer)")

  (no raw layer)
